In [ ]:
# LCEL(랭체인 익스프레션 렝귀지) : 랭체인에서 복잡한 작업 흐름을 간단하게 만들고 관리할 수 있도록 돕는 도구

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o-mini")

from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="너는 미녀와 야수에 나오는 미녀야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕? 저는 개스톤입니다. 오늘 시간 괜찮으시면 저녁 같이 먹을까요?"),
]

model.invoke(messages)
# AIMessage라는 객체 안에 여러정보가 포함되어 있다.
# 만약 텍스트 결과만 필요하다면 StrOutputParser()를 사용하면된다. (출력파서)
# 출력파서란 언어 모델이 반환하는 결과에서 원하는 형식의 데이터를 추출하는 도구임
# StrOutputParser()는 텍스트만 추출하여 반환함
# 다른 파서틑 JSON이나 객체등 특정 형식을 처리할 수 있다.
# from langchain_core.output_parsers import JsonOutputParser
# parser = JsonOutputParser()

# from pydantic import BaseModel
# from langchain.output_parsers import PydanticOutputParser
# class Person(BaseModel):
#     name: str
#     age: int
# parser = PydanticOutputParser(pydantic_object=Person)
# chain = prompt | llm | parser
# result = chain.invoke({"input": "홍길동 30살"})
# print(result.age)  # int 타입



AIMessage(content='안녕하세요, 개스턴. 당신의 제안은 고맙지만, 저는 다른 생각이 있어요. 저는 사람의 내면을 바라보는 것이 더 중요하다고 믿어요. 진정한 아름다움은 외모가 아닌 마음에서 온다고 생각해요. 당신의 마음을 보여줄 수 있을까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 62, 'total_tokens': 132, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_918210d279', 'id': 'chatcmpl-DVUqEoYxgE4kETskh1iFWPUXMZ4Gm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d99a3-0e33-7ed1-ac12-24f8a8cde009-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 62, 'output_tokens': 70, 'total_tokens': 132, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

result = model.invoke(messages) # 이와같이 사용하면 텍스트만 출겨할 수 있다.
parser.invoke(result)


'안녕하세요, 개스톤! 제 제안은 감사하지만, 저는 여러분과의 저녁이 아닌 더 특별한 어떤 것을 찾고 싶어요. 제가 소중하게 생각하는 것들이 많거든요. 기분이 좋은 따뜻한 날이라면 함께 이야기를 나누는 것도 좋을 것 같아요. 당신은 어떤 생각을 하고 있나요?'

In [ ]:
# 앞에서 작성한 코드는 gpt 모델에서 결과를 얻고 그 결과를 StrOutputParser를 사용해서 텍스트만 추출하는 2단계가 순차로 이어짐
# 이 2단계를 링체인에서는 LCEL의 체인을 이용해 | 연산자로 한줄로 간단하게 연결 할 수 있다.
chain = model | parser
chain.invoke(messages)

'안녕하세요, 개스톤. 당신의 제안은 고맙지만, 저녁은 조금 어렵겠어요. 제가 찾고 있는 것은 사랑과 진정한 아름다움이에요. 당신이 저를 위해 준비한 특별한 것이 있다면 들어보고 싶네요. 하지만 제가 원하는 것은 진정한 마음에서 우러나오는 사랑이란 것을 잊지 말아주세요.'

In [ ]:
# 다른 캐릭터가 벨에게 제안한다는 것을 탬플릿으로 만들어 보자.
# 매면 messages의 내용을 바꿔서 테스트 할 수 도 있지만 랭체인의 프롬프트 템플릿을 이용하면 필요한 부분만 수정하여 실행 할수 있다.

from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {story}에 나오는 {character_a} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 저는 {character_b}입니다. 오늘 시간 괜찮으시면 {activity} 같이 할까요?"

prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

result = prompt_template.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

print(result)

# messages=[SystemMessage(content='너는 미녀와 야수에 나오는 미녀 역할이다. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), 
# HumanMessage(content='안녕? 저는 야수입니다. 오늘 시간 괜찮으시면 저녁 같이 할까요?', additional_kwargs={}, response_metadata={})]

messages=[SystemMessage(content='너는 미녀와 야수에 나오는 미녀 역할이다. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕? 저는 야수입니다. 오늘 시간 괜찮으시면 저녁 같이 할까요?', additional_kwargs={}, response_metadata={})]


In [5]:
chain = prompt_template | model | parser

# |는 or연산이 아니라 LangChain에서 체인을 연결하는 연산자
# prompt_template → model → parser 이렇게 체인식으로 동작함
# 앞 결과를 뒤로 넘기는 파이프 연산자
chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

'안녕하세요, 야수! 저녁 함께하는 건 정말 좋은 생각이에요. 당신과 함께하는 시간이 기대돼요. 어떤 음식을 좋아하시나요? 제가 요리해 드릴까요?'